# Conclusiones

*Lo que aprendimos, qué recomendamos y qué tener en cuenta*

## Análisis exploratorio

La distribución de `Sales` es muy sesgada hacia valores bajos, con pocos pedidos de alto valor: los
atípicos son el 11.7 % de los pedidos pero el 64 % de las ventas, así que no se eliminan. `Category`
es el factor que más explica diferencias en el valor de venta: Technology y Furniture tienen pedidos
más grandes, mientras que Office Supplies domina en volumen con pedidos de menor valor.

`Region` y `Segment` muestran diferencias leves. En el tiempo, las ventas tienen un patrón estacional
que se repite cada año (bajas en enero y febrero, altas de septiembre a diciembre) y crecen desde
2016.

## Modelo de proyección

In [ ]:
from common import load_artifact, md, usd

# Resumen que guardó model/train_model.py: mejor modelo, métricas y proyección 2018
r = load_artifact("resumen.json")
# p: prueba 2017 · rf: referencia (repetir el año anterior) · rec: proyección de 2017 a 12 meses
p, rf, rec = r["prueba"], r["referencia"], r["prueba_recursiva"]
# Las conclusiones se escriben con las cifras calculadas, así se actualizan solas
md(f"- {r['mejor_modelo']} fue el mejor modelo por validación cruzada. En 2017 —un año que no vio— su "
   f"error medio por serie fue de {usd(p['MAE'])}, un {(1 - p['MAE'] / rf['MAE']) * 100:.0f} % menor que "
   f"repetir el año anterior ({usd(rf['MAE'])}).\n"
   f"- Al proyectar 2017 completo solo con datos hasta 2016, el total anual quedó a "
   f"{rec['total_desvio_total_pct']:+.1f} % del real (la referencia, a {rf['total_desvio_total_pct']:+.1f} %), "
   f"con un error típico mensual de ±{rec['total_MAE_pct']:.0f} %.\n"
   f"- Las variables más importantes son el mes del año y las ventas del mismo mes del año anterior: el "
   f"modelo confirma que la estacionalidad es el principal motor de las ventas.\n"
   f"- Para 2018 se proyectan {usd(r['proyeccion_2018'])}, "
   f"{(r['proyeccion_2018'] / r['ventas_2017'] - 1) * 100:+.0f} % frente a 2017, con el pico entre "
   f"septiembre y diciembre.")

## Recomendaciones para el negocio

Salen de los hallazgos del EDA (sección 6 de ese capítulo), donde está el dato que respalda cada una.

In [ ]:
from common import hallazgos, load_data

# Las mismas recomendaciones del capítulo EDA, calculadas con los datos
h = hallazgos(load_data())
md("\n".join(f"{n}. **{f.Factor}:** {f.Recomendación}" for n, f in enumerate(h.itertuples(), 1)))

## Limitaciones

- Dataset de ejemplo (2014–2017) de una sola tienda: no generaliza a otros negocios.
- Solo 36 meses aptos para modelar: las métricas tienen incertidumbre y un año atípico las cambiaría.
- Sin precios, costos, campañas ni datos externos (economía, competencia): el modelo solo conoce el
  historial de ventas y el calendario.
- La proyección de 2018 extiende el crecimiento reciente; si ese ritmo cambia, el error crecerá con
  cada mes proyectado.
- A nivel de una categoría en una región el error mensual es alto; la proyección es más confiable
  para totales (por región, por categoría o de la empresa).
- El EDA es asociativo: no demuestra causalidad.